# DTMIR-Pro+ — instant dehaze results from a saved model

No training here. This notebook:
1. downloads your saved `best_model.pt`
2. downloads the SOTS dehaze dataset automatically
3. runs the model on it and saves the qualitative image grid + PSNR/SSIM/LPIPS/FID

You only need to paste **two links**: the code zip and the model checkpoint.

## 1. GPU + deps

In Kaggle: **Settings → Accelerator → GPU T4**.

In [ ]:
import shutil, subprocess, torch
print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                             capture_output=True, text=True).stdout.strip())
print('CUDA:', torch.cuda.is_available())

In [ ]:
!pip install -q torch torchvision torchmetrics torch-fidelity pyyaml pillow numpy

## 2. Paste your two links

- `CODE_URL` — your `dtmir_pro_plus.zip` link
- `MODEL_URL` — your `best_model.pt` link

In [ ]:
CODE_URL = "PASTE_CODE_ZIP_LINK_HERE"
MODEL_URL = "PASTE_BEST_MODEL_PT_LINK_HERE"

## 3. Download code + model

In [ ]:
import os, zipfile, subprocess

subprocess.run(["wget", "-q", "-O", "/kaggle/working/dtmir_pro_plus.zip", CODE_URL], check=True)
with zipfile.ZipFile("/kaggle/working/dtmir_pro_plus.zip", 'r') as z:
    z.extractall('/kaggle/working')

subprocess.run(["wget", "-q", "-O", "/kaggle/working/dtmir_pro_plus/checkpoint.pt", MODEL_URL], check=True)
print('code + model downloaded')

In [ ]:
import os
os.chdir('/kaggle/working/dtmir_pro_plus')
!python sanity_check.py

## 4. Download + prepare SOTS dehaze

In [ ]:
import pathlib, subprocess, sys

url = "https://huggingface.co/datasets/desonglll/reside-SOTS/resolve/main/SOTS.zip"
if not os.path.exists("SOTS.zip"):
    print("Downloading SOTS.zip (~435 MB)...")
    subprocess.run(["wget", "-q", "-O", "SOTS.zip", url], check=True)
if not os.path.exists("sots_raw"):
    subprocess.run(["unzip", "-q", "-o", "SOTS.zip", "-d", "sots_raw"], check=True)

base = pathlib.Path("sots_raw/SOTS")
split = "outdoor" if (base / "outdoor").is_dir() else "indoor"
print("Using SOTS split:", split)

r = subprocess.run([sys.executable, "-m", "data.prepare_sots",
                    "--sots-root", str(base / split), "--outdir", "data/sots",
                    "--size", "256", "--num", "400", "--seed", "0"],
                    capture_output=True, text=True)
print(r.stdout); print(r.stderr); r.check_returncode()

## 5. Run the model and save images + metrics

In [ ]:
!python evaluate.py --checkpoint checkpoint.pt \
    --data-root data/sots/restoration --image-size 256 \
    --outdir runs/visual_sots --num 50 --grid-rows 8

## 6. Results

Download from the Kaggle **Output** tab:
- `runs/visual_sots/qualitative.png` — dehaze comparison grid (degraded | restored | clean)
- `runs/visual_sots/metrics.csv` — PSNR / SSIM / LPIPS / FID

This gives you the dehaze visuals to share now; derain/desnow are being worked on.